# Exercise 2: Mobile Device Usage Profiling

Student Name: Segundo
Student ID: TUPM-26-1952

Operating System: Android
Target Behavior Class: 3

In [1]:
import csv
import json
import shutil
from pathlib import Path

BASE_DIR = Path.cwd()

if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR / "Exercise2_Mobile_Device"

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

STUDENT_ID = "TUPM-26-1952"

SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR = (SEED_DIGIT % 5) + 1

print("Operating System:", TARGET_OS)
print("Behavior Class:", TARGET_BEHAVIOR)

Operating System: Android
Behavior Class: 3


In [2]:
input_file = DATA_DIR / "user_behavior_dataset.csv"

all_records = []
os_records = []
selected_records = []

with open(input_file, "r", encoding="utf-8-sig",
          newline="") as file:

    reader = csv.DictReader(file)

    for row in reader:

        all_records.append(row)

        operating_system = row["Operating System"].strip()

        if operating_system == TARGET_OS:
            os_records.append(row)

            try:
                behavior = int(row["User Behavior Class"])
            except (ValueError, TypeError):
                continue

            if behavior == TARGET_BEHAVIOR:
                selected_records.append(row)

print("Total records:", len(all_records))
print("Records matching OS:", len(os_records))
print("Records matching behavior:", len(selected_records))

Total records: 700
Records matching OS: 554
Records matching behavior: 111


In [3]:
battery_values = []
data_values = []

for row in selected_records:

    try:
        battery = float(row["Battery Drain (mAh/day)"])
        data = float(row["Data Usage (MB/day)"])

        battery_values.append(battery)
        data_values.append(data)

    except (ValueError, TypeError, KeyError):
        continue

if battery_values:
    average_battery = sum(battery_values) / len(battery_values)
else:
    average_battery = 0

if data_values:
    average_data = sum(data_values) / len(data_values)
else:
    average_data = 0

print("Average Battery Drain:", round(average_battery, 2), "mAh/day")
print("Average Data Usage:", round(average_data, 2), "MB/day")

Average Battery Drain: 1521.69 mAh/day
Average Data Usage: 820.26 MB/day


In [4]:
export_fields = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

filtered_csv = OUTPUT_DIR / "filtered_mobile_users.csv"

with open(filtered_csv, "w", encoding="utf-8",
          newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=export_fields
    )

    writer.writeheader()

    for row in selected_records:
        writer.writerow({
            field: row[field]
            for field in export_fields
        })

print("Exported CSV:", filtered_csv.name)

Exported CSV: filtered_mobile_users.csv


In [5]:
with open(filtered_csv, "r", encoding="utf-8-sig",
          newline="") as file:

    reader = csv.DictReader(file)
    exported_records = list(reader)

print("Selected records:", len(selected_records))
print("Exported records:", len(exported_records))

if len(selected_records) == len(exported_records):
    print("Count verification: SUCCESS")
else:
    print("Count verification: FAILED")

Selected records: 111
Exported records: 111
Count verification: SUCCESS


In [6]:
summary = {
    "student_name": "Segundo",
    "student_id": STUDENT_ID,
    "operating_system": TARGET_OS,
    "behavior_class": TARGET_BEHAVIOR,
    "total_records": len(all_records),
    "records_matching_os": len(os_records),
    "records_in_target_behavior": len(selected_records),
    "average_battery_drain": round(average_battery, 2),
    "average_data_usage": round(average_data, 2),
    "count_verification": len(selected_records) == len(exported_records)
}

json_file = OUTPUT_DIR / "mobile_usage_summary.json"

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_file.name)

JSON summary created: mobile_usage_summary.json


In [7]:
backup_file = BACKUP_DIR / "mobile_users_backup.csv"

shutil.copy2(filtered_csv, backup_file)

if backup_file.exists():
    print("Backup verification: SUCCESS")
    print("Backup file size:", backup_file.stat().st_size, "bytes")
else:
    print("Backup verification: FAILED")

Backup verification: SUCCESS
Backup file size: 4068 bytes
